In [ ]:
# --- CELL 1 ---
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# --- CELL 2 ---
IMAGE_DIR = '/content/drive/MyDrive/Oil_Spill_Project/data/train/images'
MASK_DIR = '/content/drive/MyDrive/Oil_Spill_Project/data/train/labels'
MODEL_SAVE_DIR = '/content/drive/MyDrive/Oil_Spill_Project/saved_models'
AIS_DATA_PATH = '/content/drive/MyDrive/Oil_Spill_Project/data/ais_data/vessel_data_clean.csv'

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

FINAL_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, 'sparse_unet_oil_spill.keras')

IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 40

In [ ]:
# --- CELL 3 ---
def load_image_and_mask(image_path, mask_path):
    img = tf.io.read_file(image_path)
    # Automatically handles .jpg, .jpeg, or .png
    img = tf.io.decode_image(img, channels=IMG_CHANNELS, expand_animations=False)
    img.set_shape([None, None, IMG_CHANNELS])
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

    mask = tf.io.read_file(mask_path)
    # Masks strictly stay .png to preserve 0 and 1 pixel values
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.convert_image_dtype(mask, tf.float32)
    mask = tf.image.resize(mask, [IMG_HEIGHT, IMG_WIDTH], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    mask = tf.where(mask > 0.5, 1.0, 0.0)

    return img, mask

def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    return image, mask

def get_dataset(image_dir, mask_dir, batch_size, augment_data=True):
    # Dynamically find the images whether they are jpg or png
    valid_img_exts = ('.jpg', '.jpeg', '.png')
    image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.lower().endswith(valid_img_exts)])
    mask_paths = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir) if fname.lower().endswith('.png')])

    if len(image_paths) == 0 or len(mask_paths) == 0:
        raise ValueError(f"Found {len(image_paths)} images and {len(mask_paths)} masks. Check your folders and file extensions!")
    if len(image_paths) != len(mask_paths):
        print(f"WARNING: Mismatch in count! {len(image_paths)} images vs {len(mask_paths)} masks.")

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    dataset = dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.cache()
    dataset = dataset.shuffle(buffer_size=1000)

    if augment_data:
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# Force the script to crash if paths are wrong, so we never train on fake data again.
train_dataset = get_dataset(IMAGE_DIR, MASK_DIR, BATCH_SIZE, augment_data=True)
print("✅ Dataset successfully loaded with real images!")

In [ ]:
# --- CELL 4 ---
import tensorflow.keras.backend as K

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - ((2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth))

def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)

    y_pred_f = K.clip(y_pred_f, K.epsilon(), 1.0 - K.epsilon())

    bce = tf.keras.losses.binary_crossentropy(y_true_f, y_pred_f)
    bce_exp = K.exp(-bce)
    focal_loss = alpha * K.pow((1 - bce_exp), gamma) * bce
    return K.mean(focal_loss)

def sparse_focal_dice_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred) + focal_loss(y_true, y_pred)

In [ ]:
# --- CELL 5 ---
def sparse_conv_block(x, filters, groups=4):
    # RESTORED: Your original Depthwise Separable convolutions (Spatial Sparsity)
    x = layers.SeparableConv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # RESTORED: Your original Grouped Convolutions (Channel Sparsity)
    x = layers.Conv2D(filters, 1, groups=groups, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def build_sparse_unet(input_shape=(256, 256, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)

    e1 = sparse_conv_block(inputs, 32, groups=4)
    p1 = layers.MaxPooling2D(2)(e1)

    e2 = sparse_conv_block(p1, 64, groups=4)
    p2 = layers.MaxPooling2D(2)(e2)

    e3 = sparse_conv_block(p2, 128, groups=4)
    p3 = layers.MaxPooling2D(2)(e3)

    # Replaced toxic L1 regularizers with healthy SpatialDropout
    b = sparse_conv_block(p3, 256, groups=8)
    b = layers.SpatialDropout2D(0.3)(b)

    d1 = layers.UpSampling2D(2)(b)
    d1 = layers.Concatenate()([d1, e3])
    d1 = sparse_conv_block(d1, 128, groups=4)

    d2 = layers.UpSampling2D(2)(d1)
    d2 = layers.Concatenate()([d2, e2])
    d2 = sparse_conv_block(d2, 64, groups=4)

    d3 = layers.UpSampling2D(2)(d2)
    d3 = layers.Concatenate()([d3, e1])
    d3 = sparse_conv_block(d3, 32, groups=4)

    outputs = layers.Conv2D(num_classes, 1, activation='sigmoid')(d3)

    return models.Model(inputs, outputs, name="True_Sparse_UNet")

model = build_sparse_unet((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=sparse_focal_dice_loss,
              metrics=[tf.keras.metrics.BinaryIoU(name='iou'), 'accuracy'])
model.summary()

In [ ]:
# --- CELL 6 ---
callbacks = [
    EarlyStopping(monitor='loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint(FINAL_MODEL_PATH, monitor='loss', save_best_only=True, verbose=1)
]

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# --- CELL 7 ---
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss (Sparse Focal + Dice)')
plt.title('Mathematical Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['iou'], label='Train IoU')
plt.title('Intersection Over Union Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.legend()

plt.show()

In [ ]:
# --- CELL 8 ---
def load_and_filter_ais(ais_path, bounds, lat_col='LAT', lon_col='LON'):
    if not os.path.exists(ais_path):
        return None

    df = pd.read_csv(ais_path)
    min_lat, max_lat, min_lon, max_lon = bounds

    filtered_df = df[
        (df[lat_col] >= min_lat) & (df[lat_col] <= max_lat) &
        (df[lon_col] >= min_lon) & (df[lon_col] <= max_lon)
    ]
    return filtered_df

sar_image_bounds = (28.0, 29.0, -91.0, -90.0)
nearby_vessels = load_and_filter_ais(AIS_DATA_PATH, sar_image_bounds)

if nearby_vessels is not None and not nearby_vessels.empty:
    display(nearby_vessels.head())